In [21]:
import pandas as pd
import re
from pathlib import Path

# =========================
# CONFIG
# =========================
MODE = "local"  # "local" or "google"

if MODE == "local":
    FILE_PATH = r"ac-test.xlsx"  # local workbook in repo root
    OUTPUT_CSV = r"new-template.csv"  # local output CSV in repo root
    SHARED_PATH = r"."
elif MODE == "google":
    FILE_PATH = r"drive/MyDrive/dream/rates/wip/ac-test.xlsx"
    OUTPUT_CSV = r"drive/MyDrive/dream/pauline/custom/new-template.csv"
    SHARED_PATH = r"drive/MyDrive/dream/rates"
else:
    raise ValueError("MODE must be 'local' or 'google'")

SECTION_TO_COL = {
    "SEDAN": "pass",
    "SUV / LIGHT COMMERCIAL": "lcv",
}

In [22]:
# Map sheet labels -> canonical output names
LOCATION_MAP = {
    # Full names (with carrier)
    "Melbourne Autocare": {"name": "MELBOURNE, VIC", "state": "VIC", "loc": "MELBOURNE"},
    "Sydney Autocare Dapto": {"name": "WOLLONGONG, NSW", "state": "NSW", "loc": "WOLLONGONG"},
    "Sydney Autocare Prestons": {"name": "SYDNEY, NSW", "state": "NSW", "loc": "SYDNEY"},
    "Brisbane Autocare": {"name": "BRISBANE, QLD", "state": "QLD", "loc": "BRISBANE"},
    "Townsville Autocare": {"name": "TOWNSVILLE, QLD", "state": "QLD", "loc": "TOWNSVILLE"},
    "Adelaide Autocare": {"name": "ADELAIDE, SA", "state": "SA", "loc": "ADELAIDE"},
    "Perth Autocare": {"name": "PERTH, WA", "state": "WA", "loc": "PERTH"},
    "Darwin Autocare": {"name": "DARWIN, NT", "state": "NT", "loc": "DARWIN"},
    "Devonport Autocare": {"name": "DEVONPORT, TAS", "state": "TAS", "loc": "DEVONPORT"},
    "Hobart Autocare": {"name": "HOBART, TAS", "state": "TAS", "loc": "HOBART"},
    "Sprint Mufflers Alice Springs": {"name": "ALICE SPRINGS, NT", "state": "NT", "loc": "ALICE SPRINGS"},
    "Sprint Mufflers - Alice Springs": {"name": "ALICE SPRINGS, NT", "state": "NT", "loc": "ALICE SPRINGS"},
    "Meteor Cairns": {"name": "CAIRNS, QLD", "state": "QLD", "loc": "CAIRNS"},
    "Meteor - Cairns": {"name": "CAIRNS, QLD", "state": "QLD", "loc": "CAIRNS"},
    "Meteor Mt Isa": {"name": "MT ISA, QLD", "state": "QLD", "loc": "MT ISA"},
    "Meteor - Mt Isa": {"name": "MT ISA, QLD", "state": "QLD", "loc": "MT ISA"},

    # Short names (without carrier)
    "Melbourne": {"name": "MELBOURNE, VIC", "state": "VIC", "loc": "MELBOURNE"},
    "Sydney": {"name": "SYDNEY, NSW", "state": "NSW", "loc": "SYDNEY"},
    "Wollongong": {"name": "WOLLONGONG, NSW", "state": "NSW", "loc": "WOLLONGONG"},
    "Brisbane": {"name": "BRISBANE, QLD", "state": "QLD", "loc": "BRISBANE"},
    "Townsville": {"name": "TOWNSVILLE, QLD", "state": "QLD", "loc": "TOWNSVILLE"},
    "Adelaide": {"name": "ADELAIDE, SA", "state": "SA", "loc": "ADELAIDE"},
    "Perth": {"name": "PERTH, WA", "state": "WA", "loc": "PERTH"},
    "Darwin": {"name": "DARWIN, NT", "state": "NT", "loc": "DARWIN"},
    "Devonport": {"name": "DEVONPORT, TAS", "state": "TAS", "loc": "DEVONPORT"},
    "Hobart": {"name": "HOBART, TAS", "state": "TAS", "loc": "HOBART"},
    "Alice Springs": {"name": "ALICE SPRINGS, NT", "state": "NT", "loc": "ALICE SPRINGS"},
    "Cairns": {"name": "CAIRNS, QLD", "state": "QLD", "loc": "CAIRNS"},
    "Mt Isa": {"name": "MT ISA, QLD", "state": "QLD", "loc": "MT ISA"},
    "Newcastle": {"name": "NEWCASTLE, NSW", "state": "NSW", "loc": "NEWCASTLE"},
}

STATE_CODES = {"NSW", "VIC", "QLD", "SA", "WA", "TAS", "NT", "ACT"}

In [23]:


# =========================
# CARRIER MAPPING
# =========================
# Pattern-based mapping (regex) - checked in order, first match wins
CARRIER_PATTERNS = [
    (r"(?i)\bA1\b", "A1"),
    (r"(?i)ace\s*car\s*carriers", "Ace Car Carriers"),
    (r"(?i)alfies\s*towing", "Alfies Towing"),
    (r"(?i)autocare\s*tas", "Autocare TAS"),
    (r"(?i)autocare", "Autocare"),
    (r"(?i)patricks?\s*goods", "Patricks Goods"),
    (r"(?i)automover", "Automover"),
    (r"(?i)broome\s*car\s*carriers", "Broome Car Carriers"),
    (r"(?i)byron\s*bay\s*towing", "Byron Bay Towing"),
    (r"(?i)cairns\s*car\s*carriers", "Cairns Car Carriers"),
    (r"(?i)capital\s*towing", "Capital Towing"),
    (r"(?i)carways", "Carways"),
    (r"(?i)\bceva\b", "Ceva"),
    (r"(?i)coastal\s*auto", "Coastal Auto"),
    (r"(?i)craig\s*glasson", "Craig Glasson"),
    (r"(?i)dynamic\s*car\s*carry", "Dynamic Car Carrying"),
    (r"(?i)easyway", "Easyway"),
    (r"(?i)\bgclh\b", "GCLH"),
    (r"(?i)\bgls\b", "GLS"),
    (r"(?i)gold\s*coast\s*car\s*carriers", "Gold Coast Car Carriers"),
    (r"(?i)gttkal", "GTTKAL"),
    (r"(?i)hazellbros", "Hazellbros"),
    (r"(?i)iat\s*transport", "IAT Transport"),
    (r"(?i)hunter\s*united", "Hunter United"),
    (r"(?i)jcs\s*express", "JCS Express"),
    (r"(?i)shipping", "Shipping"),
    (r"(?i)john\s*scobie", "John Scobie"),
    (r"(?i)jolly\s*(&|and)?\s*sons", "Jolly & Sons"),
    (r"(?i)\bla\s*car\s*carriers", "LA Car Carriers"),
    (r"(?i)lobeys?\s*transport", "Lobeys TRansport"),
    (r"(?i)mackay\s*car\s*(&|and)?\s*commercial", "Mackay Car & Commercial"),
    (r"(?i)melcrest", "Melcrest"),
    (r"(?i)\bmn\s*car\s*carriers", "MN Car Carriers"),
    (r"(?i)mick\s*reeve", "Mick Reeve Carriers"),
    (r"(?i)mt\s*andrews", "MT Andrews"),
    (r"(?i)nicks?\s*budget", "Nicks Budget"),
    (r"(?i)nicks?\s*express", "Nicks Express"),
    (r"(?i)newcastle\s*vehicle\s*transport|nvt", "Newcastle Vehicle Transporters"),
    (r"(?i)northern\s*vic\s*car\s*transport", "Northern Vic Car TRansport"),
    (r"(?i)\bonit\b", "Onit"),
    (r"(?i)prixcar\s*spokes", "Prixcar Spokes"),
    (r"(?i)prixcar", "Prixcar"),
    (r"(?i)progressive", "Progressive"),
    (r"(?i)reardons?", "Reardons"),
    (r"(?i)red\s*centre\s*towing", "Red Centre Towing"),
    (r"(?i)rowdys?", "Rowdys"),
    (r"(?i)saferun", "Saferun"),
    (r"(?i)searoad", "Searoad"),
    (r"(?i)\bsmb\b", "SMB"),
    (r"(?i)south\s*west\s*car\s*carriers", "South West Car Carriers"),
    (r"(?i)speedy", "Speedy"),
    (r"(?i)spoke\s*extras", "Spoke Extras"),
    (r"(?i)tamworth\s*car\s*carriers", "Tamworth Car Carriers"),
    (r"(?i)tge\s*car\s*carry", "TGE Car Carrying"),
    (r"(?i)tidy\s*towing", "Tidy Towing"),
    (r"(?i)top\s*end\s*car\s*carriers", "TopEnd Car Carriers"),
    (r"(?i)western\s*vehicle\s*movers", "Western Vehicle Movers"),
    (r"(?i)sim\s*cam", "Sim Cam"),
    (r"(?i)way\s*2\s*go", "Way 2 Go"),
    (r"(?i)westoz", "Westoz"),
]

DEFAULT_CARRIER = "UNKNOWN"


def get_carrier_from_sheet_name(sheet_name: str) -> str:
    """Return carrier name based on sheet name pattern matching."""
    for pattern, carrier in CARRIER_PATTERNS:
        if re.search(pattern, sheet_name):
            return carrier
    return DEFAULT_CARRIER

In [24]:
def is_text_label(x):
    """Check if value is a text label (contains letters)."""
    return pd.notna(x) and isinstance(x, str) and any(ch.isalpha() for ch in x)


def parse_money(x):
    """Parse monetary value from cell (handles $, commas, etc)."""
    if pd.isna(x):
        return None
    if isinstance(x, (int, float)):
        return float(x)
    if isinstance(x, str):
        s = x.strip().replace("$", "").replace(",", "")
        if not s:
            return None
        try:
            return float(s)
        except ValueError:
            return None
    return None


def clean_text(s):
    """Normalize whitespace in text."""
    return re.sub(r"\s+", " ", str(s)).strip()


def normalize_location(raw_label: str):
    """Convert raw location label to canonical name, state, and loc (all uppercase)."""
    raw_label = clean_text(raw_label)

    if raw_label in LOCATION_MAP:
        m = LOCATION_MAP[raw_label]
        return m["name"], m["state"], m["loc"]

    # fallback if already like "CITY, ST"
    m = re.match(r"^\s*(.+?)\s*,\s*([A-Z]{2,3})\s*$", raw_label.upper())
    if m and m.group(2) in STATE_CODES:
        city, state = m.group(1).strip().upper(), m.group(2).upper()
        return f"{city}, {state}", state, city

    # last-resort fallback (uppercase)
    return raw_label.upper(), pd.NA, raw_label.upper()

In [25]:
def parse_matrix_to_wide(path, sheet=0, debug=True):
    """Parse cross-matrix from Excel and return wide-format DataFrame."""
    raw = pd.read_excel(path, sheet_name=sheet, header=None)

    if debug:
        print("raw shape:", raw.shape)

    txt = raw.copy()
    for c in txt.columns:
        txt[c] = txt[c].astype(str).str.strip()

    records = []

    for section_name, metric in SECTION_TO_COL.items():
        section_hits = (txt == section_name).any(axis=1)
        if not section_hits.any():
            if debug:
                print(f"[WARN] section not found: {section_name}")
            continue

        srow = section_hits[section_hits].index[0]

        # In your sheet, destination headers are on SAME row as section label.
        header_cols_same_row = [c for c in range(1, raw.shape[1]) if is_text_label(raw.iat[srow, c])]
        if len(header_cols_same_row) >= 1:  # Support matrices with 1+ destinations
            hrow = srow
            valid_header_cols = header_cols_same_row
        else:
            # Fallback for other formats
            hrow = srow + 1
            valid_header_cols = [c for c in range(1, raw.shape[1]) if is_text_label(raw.iat[hrow, c])]

        if not valid_header_cols:
            if debug:
                print(f"[WARN] no destination headers for section: {section_name}")
            continue

        if debug:
            print(f"{section_name}: section_row={srow}, header_row={hrow}, destinations={len(valid_header_cols)}")

        # Data starts after header row
        r = hrow + 1
        while r < len(raw):
            row_text = " ".join(str(v).strip().upper() for v in raw.iloc[r].tolist() if pd.notna(v))
            if any(sec in row_text for sec in SECTION_TO_COL.keys()):
                break

            from_cell = raw.iat[r, 0]
            if pd.notna(from_cell) and str(from_cell).strip() != "":
                from_raw = clean_text(from_cell)

                for c in valid_header_cols:
                    to_cell = raw.iat[hrow, c]
                    val = parse_money(raw.iat[r, c])

                    if is_text_label(to_cell) and val is not None:
                        records.append({
                            "from_raw": from_raw,
                            "to_raw": clean_text(to_cell),
                            "metric": metric,
                            "value": val,
                        })
            r += 1

    if not records:
        raise ValueError("No records parsed. Check FILE_PATH/SHEET_NAME and Excel layout.")

    long_df = pd.DataFrame(records)

    wide = (
        long_df
        .pivot_table(
            index=["from_raw", "to_raw"],
            columns="metric",
            values="value",
            aggfunc="first"
        )
        .reset_index()
    )
    wide.columns.name = None

    # Ensure both columns exist
    for col in ["pass", "lcv"]:
        if col not in wide.columns:
            wide[col] = pd.NA

    # clean text/newlines
    for col in ["from_raw", "to_raw"]:
        wide[col] = wide[col].astype(str).apply(clean_text)

    return wide.reset_index(drop=True)

In [26]:
def build_target_schema(wide_df, carrier):
    """Convert wide DataFrame to target output schema (all location/carrier columns uppercase)."""
    out = wide_df.copy()

    # Filter out rows where pass or lcv is NaN
    out = out.dropna(subset=["pass", "lcv"], how="any")

    from_norm = out["from_raw"].apply(normalize_location)
    to_norm = out["to_raw"].apply(normalize_location)

    out["from"] = from_norm.apply(lambda x: x[0])
    out["from_state"] = from_norm.apply(lambda x: x[1])
    out["from_loc"] = from_norm.apply(lambda x: x[2])

    out["to"] = to_norm.apply(lambda x: x[0])
    out["to_state"] = to_norm.apply(lambda x: x[1])
    out["to_loc"] = to_norm.apply(lambda x: x[2])

    out["carrier"] = carrier.upper()
    out["tab"] = "hub"
    out["carrier_type"] = "hub"
    out["transit"] = pd.NA

    # Split into same-origin and different-origin
    same_origin = out[out["from"] == out["to"]].copy()
    diff_origin = out[out["from"] != out["to"]].copy()

    # Different origin: depot-depot (single record)
    diff_origin["type"] = "depot-depot"

    # Same origin: create depot-door and door-depot records
    if not same_origin.empty:
        depot_door = same_origin.copy()
        depot_door["type"] = "depot-door"

        door_depot = same_origin.copy()
        door_depot["type"] = "door-depot"

        same_origin_records = pd.concat([depot_door, door_depot], ignore_index=True)
    else:
        same_origin_records = pd.DataFrame()

    # Combine all records
    final = pd.concat([diff_origin, same_origin_records], ignore_index=True)

    return final[
        [
            "from", "to", "from_state", "to_state",
            "pass", "lcv",
            "type", "carrier", "tab", "carrier_type", "transit",
            "from_loc", "to_loc",
        ]
    ]

In [27]:
def get_sheet_names(path):
    """Return list of all sheet names in workbook."""
    xl = pd.ExcelFile(path)
    return xl.sheet_names


def print_unmapped_labels(wide_df):
    """Print any location labels not in LOCATION_MAP."""
    labels = set(wide_df["from_raw"]) | set(wide_df["to_raw"])
    missing = sorted([x for x in labels if x not in LOCATION_MAP])
    if missing:
        print("\n[WARN] Unmapped labels (add to LOCATION_MAP):")
        for m in missing:
            print(f"  - {m}")
    else:
        print("\n[INFO] All labels mapped.")

In [28]:
def parse_all_sheets(path, debug=True):
    """Parse all sheets and return dictionary keyed by sheet name."""
    sheet_names = get_sheet_names(path)
    results = {}
    unmapped_carriers = []

    for sheet_name in sheet_names:
        if debug:
            print(f"\n{'='*50}")
            print(f"Processing sheet: {sheet_name}")
            print('='*50)

        try:
            wide = parse_matrix_to_wide(path, sheet=sheet_name, debug=debug)

            # Get carrier for this sheet
            carrier = get_carrier_from_sheet_name(sheet_name)
            if carrier == DEFAULT_CARRIER:
                unmapped_carriers.append(sheet_name)
                if debug:
                    print(f"[WARN] Carrier: {carrier} (unmapped - add pattern to CARRIER_PATTERNS)")
            else:
                if debug:
                    print(f"[INFO] Carrier: {carrier}")

            final_df = build_target_schema(wide, carrier=carrier)
            results[sheet_name] = final_df

            if debug:
                print(f"[OK] {sheet_name}: {len(final_df)} rows")
                print_unmapped_labels(wide)
        except Exception as e:
            if debug:
                print(f"[SKIP] {sheet_name}: {e}")
            continue

    # Final summary of unmapped carriers
    if unmapped_carriers and debug:
        print(f"\n{'='*50}")
        print(f"[WARN] UNMAPPED CARRIERS SUMMARY")
        print('='*50)
        print(f"The following sheets could not be mapped to a carrier:")
        for sheet in unmapped_carriers:
            print(f"  - {sheet}")
        print(f"\nAdd patterns to CARRIER_PATTERNS to map these sheets.")

    return results

In [29]:
# =========================
# RUN - Parse All Sheets
# =========================
all_sheets = parse_all_sheets(FILE_PATH, debug=True)

print(f"\n{'='*50}")
print(f"SUMMARY: Parsed {len(all_sheets)} sheets")
print('='*50)

unmapped_carriers = []
for sheet_name, df in all_sheets.items():
    carrier = df['carrier'].iloc[0]
    print(f"  - {sheet_name}: {len(df)} rows (carrier: {carrier})")
    if carrier == DEFAULT_CARRIER:
        unmapped_carriers.append(sheet_name)

# Final warning about unmapped carriers
if unmapped_carriers:
    print(f"\n[WARN] Sheets with unmapped carrier (using '{DEFAULT_CARRIER}'):")
    for sheet in unmapped_carriers:
        print(f"  - {sheet}")
    print("\nAdd patterns to CARRIER_PATTERNS to map these sheets.")
else:
    print("\n[INFO] All sheets mapped to a carrier.")


Processing sheet: Autocare Rate Matrix 14012026
raw shape: (48, 14)
SEDAN: section_row=4, header_row=4, destinations=13
SUV / LIGHT COMMERCIAL: section_row=19, header_row=19, destinations=13
[INFO] Carrier: Autocare
[OK] Autocare Rate Matrix 14012026: 156 rows

[WARN] Unmapped labels (add to LOCATION_MAP):
  - Mount Isa

Processing sheet: Shipping
raw shape: (35, 11)
SEDAN: section_row=1, header_row=1, destinations=10
SUV / LIGHT COMMERCIAL: section_row=11, header_row=11, destinations=10
[INFO] Carrier: Shipping
[OK] Shipping: 11 rows

[INFO] All labels mapped.

Processing sheet: Automover
raw shape: (27, 6)
SEDAN: section_row=1, header_row=1, destinations=5
SUV / LIGHT COMMERCIAL: section_row=7, header_row=7, destinations=5
[INFO] Carrier: Automover
[OK] Automover: 11 rows

[INFO] All labels mapped.

Processing sheet: A1
raw shape: (25, 4)
SEDAN: section_row=1, header_row=1, destinations=3
SUV / LIGHT COMMERCIAL: section_row=6, header_row=6, destinations=3
[INFO] Carrier: A1
[OK] A1:

In [30]:
# =========================
# PREVIEW - Display each sheet
# =========================
for sheet_name, df in all_sheets.items():
    print(f"\n{'='*50}")
    print(f"Sheet: {sheet_name}")
    print('='*50)
    display(df.head(10))


Sheet: Autocare Rate Matrix 14012026


,from,to,from_state,to_state,pass,lcv,type,carrier,tab,carrier_type,transit,from_loc,to_loc
0,"ADELAIDE, SA","ALICE SPRINGS, NT",SA,NT,950.0,1187.50,depot-depot,AUTOCARE,hub,hub,<NA>,ADELAIDE,ALICE SPRINGS
1,"ADELAIDE, SA","BRISBANE, QLD",SA,QLD,1029.0,1286.25,depot-depot,AUTOCARE,hub,hub,<NA>,ADELAIDE,BRISBANE
2,"ADELAIDE, SA","CAIRNS, QLD",SA,QLD,1856.0,2320.00,depot-depot,AUTOCARE,hub,hub,<NA>,ADELAIDE,CAIRNS
3,"ADELAIDE, SA","DARWIN, NT",SA,NT,1060.8,1644.24,depot-depot,AUTOCARE,hub,hub,<NA>,ADELAIDE,DARWIN
4,"ADELAIDE, SA","DEVONPORT, TAS",SA,TAS,728.8,911.00,depot-depot,AUTOCARE,hub,hub,<NA>,ADELAIDE,DEVONPORT
5,"ADELAIDE, SA","HOBART, TAS",SA,TAS,1126.0,1407.50,depot-depot,AUTOCARE,hub,hub,<NA>,ADELAIDE,HOBART
6,"ADELAIDE, SA","MELBOURNE, VIC",SA,VIC,218.4,273.00,depot-depot,AUTOCARE,hub,hub,<NA>,ADELAIDE,MELBOURNE
7,"ADELAIDE, SA",MOUNT ISA,SA,NaN,2321.0,2901.25,depot-depot,AUTOCARE,hub,hub,<NA>,ADELAIDE,MOUNT ISA
8,"ADELAIDE, SA","PERTH, WA",SA,WA,838.0,1047.50,depot-depot,AUTOCARE,hub,hub,<NA>,ADELAIDE,PERTH
9,"ADELAIDE, SA","SYDNEY, NSW",SA,NSW,680.0,850.00,depot-depot,AUTOCARE,hub,hub,<NA>,ADELAIDE,SYDNEY



Sheet: Shipping


,from,to,from_state,to_state,pass,lcv,type,carrier,tab,carrier_type,transit,from_loc,to_loc
0,"ADELAIDE, SA","PERTH, WA",SA,WA,1227.27,1318.18,depot-depot,SHIPPING,hub,hub,<NA>,ADELAIDE,PERTH
1,"BRISBANE, QLD","PERTH, WA",QLD,WA,1177.27,1268.18,depot-depot,SHIPPING,hub,hub,<NA>,BRISBANE,PERTH
2,"DEVONPORT, TAS","PERTH, WA",TAS,WA,1854.00,1954.00,depot-depot,SHIPPING,hub,hub,<NA>,DEVONPORT,PERTH
3,"HOBART, TAS","PERTH, WA",TAS,WA,2043.00,2143.00,depot-depot,SHIPPING,hub,hub,<NA>,HOBART,PERTH
4,"MELBOURNE, VIC","PERTH, WA",VIC,WA,1250.00,1318.18,depot-depot,SHIPPING,hub,hub,<NA>,MELBOURNE,PERTH
5,"PERTH, WA","BRISBANE, QLD",WA,QLD,1341.73,1441.73,depot-depot,SHIPPING,hub,hub,<NA>,PERTH,BRISBANE
6,"PERTH, WA","MELBOURNE, VIC",WA,VIC,1073.70,1173.70,depot-depot,SHIPPING,hub,hub,<NA>,PERTH,MELBOURNE
7,"PERTH, WA","SYDNEY, NSW",WA,NSW,1204.42,1304.42,depot-depot,SHIPPING,hub,hub,<NA>,PERTH,SYDNEY
8,"PERTH, WA","WOLLONGONG, NSW",WA,NSW,1100.00,1200.00,depot-depot,SHIPPING,hub,hub,<NA>,PERTH,WOLLONGONG
9,"SYDNEY, NSW","PERTH, WA",NSW,WA,1336.36,1427.27,depot-depot,SHIPPING,hub,hub,<NA>,SYDNEY,PERTH



Sheet: Automover


...

,from,to,from_state,to_state,pass,lcv,type,carrier,tab,carrier_type,transit,from_loc,to_loc
0,"BRISBANE, QLD","CAIRNS, QLD",QLD,QLD,1150.0,1150.0,depot-depot,IAT TRANSPORT,hub,hub,<NA>,BRISBANE,CAIRNS
1,"BRISBANE, QLD","TOWNSVILLE, QLD",QLD,QLD,900.0,900.0,depot-depot,IAT TRANSPORT,hub,hub,<NA>,BRISBANE,TOWNSVILLE
2,"CAIRNS, QLD","BRISBANE, QLD",QLD,QLD,1150.0,1150.0,depot-depot,IAT TRANSPORT,hub,hub,<NA>,CAIRNS,BRISBANE
3,"TOWNSVILLE, QLD","BRISBANE, QLD",QLD,QLD,800.0,800.0,depot-depot,IAT TRANSPORT,hub,hub,<NA>,TOWNSVILLE,BRISBANE



Sheet: GTTKAL


,from,to,from_state,to_state,pass,lcv,type,carrier,tab,carrier_type,transit,from_loc,to_loc
0,KALGOORLIE,"PERTH, WA",NaN,WA,3422.1,3422.1,depot-depot,GTTKAL,hub,hub,<NA>,KALGOORLIE,PERTH
1,"PERTH, WA",KALGOORLIE,WA,NaN,3422.1,3442.1,depot-depot,GTTKAL,hub,hub,<NA>,PERTH,KALGOORLIE
2,KALGOORLIE,KALGOORLIE,NaN,NaN,195.2,195.2,depot-door,GTTKAL,hub,hub,<NA>,KALGOORLIE,KALGOORLIE
3,KALGOORLIE,KALGOORLIE,NaN,NaN,195.2,195.2,door-depot,GTTKAL,hub,hub,<NA>,KALGOORLIE,KALGOORLIE



Sheet: Newcastle Vehicle Transporters


,from,to,from_state,to_state,pass,lcv,type,carrier,tab,carrier_type,transit,from_loc,to_loc
0,"NEWCASTLE, NSW","SYDNEY, NSW",NSW,NSW,400.0,500.0,depot-depot,NEWCASTLE VEHICLE TRANSPORTERS,hub,hub,<NA>,NEWCASTLE,SYDNEY
1,"SYDNEY, NSW","NEWCASTLE, NSW",NSW,NSW,400.0,500.0,depot-depot,NEWCASTLE VEHICLE TRANSPORTERS,hub,hub,<NA>,SYDNEY,NEWCASTLE


In [31]:
# =========================
# CONCATENATE ALL SHEETS INTO ONE DATAFRAME
# =========================
all_data = pd.concat(
    [df.assign(sheet=name) for name, df in all_sheets.items()],
    ignore_index=True
)

all_data['tab'] = 'new-template'
del(all_data['sheet'])

print(f"Total rows: {len(all_data)}")
display(all_data)

if OUTPUT_CSV:
  all_data.to_csv(OUTPUT_CSV, index = False)

Total rows: 316


,from,to,from_state,to_state,pass,lcv,type,carrier,tab,carrier_type,transit,from_loc,to_loc
0,"ADELAIDE, SA","ALICE SPRINGS, NT",SA,NT,950.0,1187.50,depot-depot,AUTOCARE,new-template,hub,<NA>,ADELAIDE,ALICE SPRINGS
1,"ADELAIDE, SA","BRISBANE, QLD",SA,QLD,1029.0,1286.25,depot-depot,AUTOCARE,new-template,hub,<NA>,ADELAIDE,BRISBANE
2,"ADELAIDE, SA","CAIRNS, QLD",SA,QLD,1856.0,2320.00,depot-depot,AUTOCARE,new-template,hub,<NA>,ADELAIDE,CAIRNS
3,"ADELAIDE, SA","DARWIN, NT",SA,NT,1060.8,1644.24,depot-depot,AUTOCARE,new-template,hub,<NA>,ADELAIDE,DARWIN
4,"ADELAIDE, SA","DEVONPORT, TAS",SA,TAS,728.8,911.00,depot-depot,AUTOCARE,new-template,hub,<NA>,ADELAIDE,DEVONPORT
...,...,...,...,...,...,...,...,...,...,...,...,...,...
311,"PERTH, WA",KALGOORLIE,WA,NaN,3422.1,3442.10,depot-depot,GTTKAL,new-template,hub,<NA>,PERTH,KALGOORLIE
312,KALGOORLIE,KALGOORLIE,NaN,NaN,195.2,195.20,depot-door,GTTKAL,new-template,hub,<NA>,KALGOORLIE,KALGOORLIE
313,KALGOORLIE,KALGOORLIE,NaN,NaN,195.2,195.20,door-depot,GTTKAL,new-template,hub,<NA>,KALGOORLIE,KALGOORLIE
314,"NEWCASTLE, NSW","SYDNEY, NSW",NSW,NSW,400.0,500.00,depot-depot,NEWCASTLE VEHICLE TRANSPORTERS,new-template,hub,<NA>,NEWCASTLE,SYDNEY


In [32]:
all_data

,from,to,from_state,to_state,pass,lcv,type,carrier,tab,carrier_type,transit,from_loc,to_loc
0,"ADELAIDE, SA","ALICE SPRINGS, NT",SA,NT,950.0,1187.50,depot-depot,AUTOCARE,new-template,hub,<NA>,ADELAIDE,ALICE SPRINGS
1,"ADELAIDE, SA","BRISBANE, QLD",SA,QLD,1029.0,1286.25,depot-depot,AUTOCARE,new-template,hub,<NA>,ADELAIDE,BRISBANE
2,"ADELAIDE, SA","CAIRNS, QLD",SA,QLD,1856.0,2320.00,depot-depot,AUTOCARE,new-template,hub,<NA>,ADELAIDE,CAIRNS
3,"ADELAIDE, SA","DARWIN, NT",SA,NT,1060.8,1644.24,depot-depot,AUTOCARE,new-template,hub,<NA>,ADELAIDE,DARWIN
4,"ADELAIDE, SA","DEVONPORT, TAS",SA,TAS,728.8,911.00,depot-depot,AUTOCARE,new-template,hub,<NA>,ADELAIDE,DEVONPORT
...,...,...,...,...,...,...,...,...,...,...,...,...,...
311,"PERTH, WA",KALGOORLIE,WA,NaN,3422.1,3442.10,depot-depot,GTTKAL,new-template,hub,<NA>,PERTH,KALGOORLIE
312,KALGOORLIE,KALGOORLIE,NaN,NaN,195.2,195.20,depot-door,GTTKAL,new-template,hub,<NA>,KALGOORLIE,KALGOORLIE
313,KALGOORLIE,KALGOORLIE,NaN,NaN,195.2,195.20,door-depot,GTTKAL,new-template,hub,<NA>,KALGOORLIE,KALGOORLIE
314,"NEWCASTLE, NSW","SYDNEY, NSW",NSW,NSW,400.0,500.00,depot-depot,NEWCASTLE VEHICLE TRANSPORTERS,new-template,hub,<NA>,NEWCASTLE,SYDNEY
